# Classical Models — No Title Normalization

This notebook replicates the classical modelling pipeline from `Final_Headline_Project.ipynb` with **one change**: Section 3.4 (title normalisation) is omitted. `title_clean` is set equal to the raw `title` column so that all downstream code is unchanged. The goal is to compare classical-model performance with and without normalisation.

# 0. Environment Setup

In [1]:
import re, random, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import spacy
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import textstat
from datasketch import MinHash, MinHashLSH
import scipy.sparse as sp
import pickle
import warnings

from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              average_precision_score, brier_score_loss)
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, TransformerMixin

SEEDS      = [42, 123, 456]
DATA_DIR   = Path("Datasets")
OUTPUT_DIR = Path("outputs_no_norm");  OUTPUT_DIR.mkdir(exist_ok=True)
MODELS_DIR = Path("models_no_norm");   MODELS_DIR.mkdir(exist_ok=True)

random.seed(SEEDS[0])
np.random.seed(SEEDS[0])

print("Environment ready.")
print(f"  Output : {OUTPUT_DIR.resolve()}")
print(f"  Models : {MODELS_DIR.resolve()}")

/Users/lukastefanovic/ds2_env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Environment ready.
  Output : /Users/lukastefanovic/Desktop/Data Science II/outputs_no_norm
  Models : /Users/lukastefanovic/Desktop/Data Science II/models_no_norm


# 2. Data Loading

## 2.1 WELFake

In [2]:
# Schema inspection; WELFake already uses 0=Real/1=Fake so harmonize_labels is a no-op.
welfake_raw = pd.read_csv(DATA_DIR / "WELFake_Dataset.csv")

print("Shape:", welfake_raw.shape)
print("\nColumns:", welfake_raw.columns.tolist())
print("\nMissing values:\n", welfake_raw.isnull().sum())
print("\nLabel distribution (0=Real, 1=Fake):")
vc  = welfake_raw["label"].value_counts()
pct = welfake_raw["label"].value_counts(normalize=True).mul(100).round(2)
print(pd.DataFrame({"count": vc, "pct%": pct}))

Shape: (72134, 4)

Columns: ['Unnamed: 0', 'title', 'text', 'label']

Missing values:
 Unnamed: 0      0
title         558
text           39
label           0
dtype: int64

Label distribution (0=Real, 1=Fake):
       count   pct%
label              
1      37106  51.44
0      35028  48.56


## 2.2 ISOT

In [3]:
# ISOT ships as two files; tag source_file before concat so harmonize_labels can derive the label.
isot_true = pd.read_csv(DATA_DIR / "News-_dataset" / "True.csv")
isot_fake = pd.read_csv(DATA_DIR / "News-_dataset" / "Fake.csv")

isot_true["raw_label"]   = 0;  isot_true["source_file"] = "True.csv"
isot_fake["raw_label"]   = 1;  isot_fake["source_file"] = "Fake.csv"

isot_raw = pd.concat([isot_true, isot_fake], ignore_index=True)

print("isot_raw shape:", isot_raw.shape)
print("\nMissing values:\n", isot_raw.isnull().sum())

isot_raw shape: (44898, 6)

Missing values:
 title          0
text           0
subject        0
date           0
raw_label      0
source_file    0
dtype: int64


## 2.3 EU_mix

In [4]:
eu_mix_path = DATA_DIR / "thrid dataset" / "EU_mix_3_EMNAD_final.txt"

print("First 5 raw lines:")
with open(eu_mix_path, encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5: break
        print(repr(line))

First 5 raw lines:
'title,text,label,Unnamed: 0,Date,URL,Outlets,Countries,Language,Disproof,Keywords\n'
'Saints search for manager with ‘long-term vision’,"Saints search for manager with ‘long-term vision’\n'
'\n'
'Southampton are looking for a manager with a shared “long-term vision” after sacking Claude Puel.\n'
'Frenchman Puel, 55, took over at St Mary’s last summer and guided Saints to the EFL Cup final, where they lost to Manchester United, and an eighth-placed finish in last season’s Premier League.\n'


In [5]:
# raw-line check above confirms comma-separated
eu_mix_raw = pd.read_csv(eu_mix_path, sep=",", encoding="utf-8")

print("Shape:", eu_mix_raw.shape)
print("\nColumns:", eu_mix_raw.columns.tolist())
print("\nMissing values:\n", eu_mix_raw.isnull().sum())

Shape: (19930, 11)

Columns: ['title', 'text', 'label', 'Unnamed: 0', 'Date', 'URL', 'Outlets', 'Countries', 'Language', 'Disproof', 'Keywords']

Missing values:
 title             0
text              2
label             0
Unnamed: 0     9965
Date           9965
URL            9965
Outlets       10154
Countries     10082
Language       9967
Disproof       9967
Keywords      10367
dtype: int64


## 2.4 Label Convention

In [6]:
# Normalise to 0=Real/1=Fake across all three corpora.
def harmonize_labels(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    df = df.copy()
    if dataset_name == "isot":
        df["label"] = df["source_file"].map({"True.csv": 0, "Fake.csv": 1})
    return df

welfake_loaded = harmonize_labels(welfake_raw, "welfake")
isot_loaded    = harmonize_labels(isot_raw,    "isot")
eu_mix_loaded  = harmonize_labels(eu_mix_raw,  "eu_mix")

for name, df in [("WELFake", welfake_loaded), ("ISOT", isot_loaded), ("EU_mix", eu_mix_loaded)]:
    print(f"{name} label distribution (0=Real, 1=Fake):")
    vc  = df["label"].value_counts()
    pct = df["label"].value_counts(normalize=True).mul(100).round(2)
    print(pd.DataFrame({"count": vc, "pct%": pct}))
    print()

WELFake label distribution (0=Real, 1=Fake):
       count   pct%
label              
1      37106  51.44
0      35028  48.56

ISOT label distribution (0=Real, 1=Fake):
       count  pct%
label             
1      23481  52.3
0      21417  47.7

EU_mix label distribution (0=Real, 1=Fake):
       count  pct%
label             
1       9965  50.0
0       9965  50.0



# 3. Data Cleaning

## 3.1 WELFake

In [7]:
df = welfake_loaded.copy()
print(f"WELFake starting rows: {len(df)}")

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

n = len(df)
df = df.dropna(subset=["title"])
print(f"Dropped {n - len(df)} null titles  → {len(df)} rows")

n = len(df)
df = df.dropna(subset=["text"])
print(f"Dropped {n - len(df)} null text rows → {len(df)} rows")

n = len(df)
df = df.drop_duplicates(subset=["title"], keep="first")
print(f"Dropped {n - len(df)} duplicate titles → {len(df)} rows")

welfake_clean = df.reset_index(drop=True)
print(f"\nwelfake_clean: {welfake_clean.shape}")

WELFake starting rows: 72134
Dropped 558 null titles  → 71576 rows
Dropped 39 null text rows → 71537 rows
Dropped 9229 duplicate titles → 62308 rows

welfake_clean: (62308, 3)


## 3.2 ISOT

In [8]:
df = isot_loaded.copy()
print(f"ISOT starting rows: {len(df)}")

drop_cols = ["subject", "date", "raw_label", "source_file"]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

n = len(df)
df = df.drop_duplicates(subset=["title"], keep="first")
print(f"Dropped {n - len(df)} duplicate titles → {len(df)} rows")

isot_clean = df.reset_index(drop=True)
print(f"\nisot_clean: {isot_clean.shape}")

ISOT starting rows: 44898
Dropped 6169 duplicate titles → 38729 rows

isot_clean: (38729, 3)


## 3.3 EU_mix

In [9]:
df = eu_mix_loaded.copy()
print(f"EU_mix starting rows: {len(df)}")

meta_cols = ["Unnamed: 0", "Date", "URL", "Outlets", "Countries", "Language", "Disproof", "Keywords"]
df = df.drop(columns=[c for c in meta_cols if c in df.columns])

n = len(df)
df = df.dropna(subset=["text"])
print(f"Dropped {n - len(df)} null text rows → {len(df)} rows")

n = len(df)
df = df.drop_duplicates(subset=["title"], keep="first")
print(f"Dropped {n - len(df)} duplicate titles → {len(df)} rows")

eu_mix_clean = df.reset_index(drop=True)
print(f"\neu_mix_clean: {eu_mix_clean.shape}")

EU_mix starting rows: 19930
Dropped 2 null text rows → 19928 rows
Dropped 101 duplicate titles → 19827 rows

eu_mix_clean: (19827, 3)


## 3.4 Title Normalization — SKIPPED

> **Experiment:** Section 3.4 from the main notebook (lowercase, URL removal, attribution stripping) is intentionally omitted. `title_clean` is set to the raw `title` so all downstream code runs unchanged. This lets us measure whether normalisation materially affects classical-model F1.

In [10]:
# Skipping normalisation: assign raw title as title_clean for all three corpora.
for df in [welfake_clean, isot_clean, eu_mix_clean]:
    df["title_clean"] = df["title"]

print("title_clean set to raw title (no normalisation).")
print(f"  welfake_clean example: {welfake_clean['title_clean'].iloc[0]}")
print(f"  isot_clean    example: {isot_clean['title_clean'].iloc[0]}")
print(f"  eu_mix_clean  example: {eu_mix_clean['title_clean'].iloc[0]}")

title_clean set to raw title (no normalisation).
  welfake_clean example: LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #BlackLivesMatter And #FYF911 Terrorists [VIDEO]
  isot_clean    example: As U.S. budget fight looms, Republicans flip their fiscal script
  eu_mix_clean  example: Saints search for manager with ‘long-term vision’


## 3.5 Save Cleaned Datasets

In [11]:
datasets = [
    ("welfake_clean", welfake_clean),
    ("isot_clean",    isot_clean),
    ("eu_mix_clean",  eu_mix_clean),
]

for name, df in datasets:
    path = OUTPUT_DIR / f"{name}.csv"
    df.to_csv(path, index=False, encoding="utf-8")
    reloaded = pd.read_csv(path, encoding="utf-8")
    assert len(reloaded) == len(df), f"Row count mismatch for {name}"

print("Provenance summary:")
print(f"{'Dataset':<16} {'Rows':>7}  Columns")
print("-" * 55)
for name, df in datasets:
    print(f"{name:<16} {len(df):>7}  {df.columns.tolist()}")
print(f"\nAll files written to: {OUTPUT_DIR.resolve()}")

Provenance summary:
Dataset             Rows  Columns
-------------------------------------------------------
welfake_clean      62308  ['title', 'text', 'label', 'title_clean']
isot_clean         38729  ['title', 'text', 'label', 'title_clean']
eu_mix_clean       19827  ['title', 'text', 'label', 'title_clean']

All files written to: /Users/lukastefanovic/Desktop/Data Science II/outputs_no_norm


# 4. Leakage Audit

## 4.1 Exact Match Overlap

In [12]:
wf_titles    = set(welfake_clean["title_clean"].dropna())
isot_titles  = set(isot_clean["title_clean"].dropna())
eumix_titles = set(eu_mix_clean["title_clean"].dropna())

exact_overlap_isot  = wf_titles & isot_titles
exact_overlap_eumix = wf_titles & eumix_titles

n_wf = len(welfake_clean)
print(f"WELFake × ISOT   exact matches: {len(exact_overlap_isot):,}  ({100*len(exact_overlap_isot)/n_wf:.3f}% of WELFake)")
print(f"WELFake × EU_mix exact matches: {len(exact_overlap_eumix):,}  ({100*len(exact_overlap_eumix)/n_wf:.3f}% of WELFake)")

WELFake × ISOT   exact matches: 38,729  (62.157% of WELFake)
WELFake × EU_mix exact matches: 10  (0.016% of WELFake)


## 4.2 Near-Duplicate Overlap via MinHash

In [13]:
NUM_PERM  = 128
THRESHOLD = 0.9

def make_minhash(text, num_perm=NUM_PERM):
    m = MinHash(num_perm=num_perm)
    words = str(text).split()
    if len(words) >= 3:
        shingles = {" ".join(words[i:i+3]) for i in range(len(words) - 2)}
    else:
        shingles = set(words)
    for s in shingles:
        m.update(s.encode("utf-8"))
    return m

lsh = MinHashLSH(threshold=THRESHOLD, num_perm=NUM_PERM)

print("Inserting ISOT titles...")
for i, title in enumerate(isot_clean["title_clean"]):
    lsh.insert(f"isot_{i}", make_minhash(title))

print("Inserting EU_mix titles...")
for i, title in enumerate(eu_mix_clean["title_clean"]):
    lsh.insert(f"eumix_{i}", make_minhash(title))

print(f"Querying {len(welfake_clean):,} WELFake titles...")
fuzzy_overlap_isot  = set()
fuzzy_overlap_eumix = set()

for title in welfake_clean["title_clean"]:
    results_lsh = lsh.query(make_minhash(title))
    for r in results_lsh:
        if r.startswith("isot_"):
            fuzzy_overlap_isot.add(title)
        elif r.startswith("eumix_"):
            fuzzy_overlap_eumix.add(title)

print(f"\nWELFake × ISOT   fuzzy (J≥0.9): {len(fuzzy_overlap_isot):,}")
print(f"WELFake × EU_mix fuzzy (J≥0.9): {len(fuzzy_overlap_eumix):,}")

Inserting ISOT titles...
Inserting EU_mix titles...
Querying 62,308 WELFake titles...

WELFake × ISOT   fuzzy (J≥0.9): 38,737
WELFake × EU_mix fuzzy (J≥0.9): 11


## 4.3 Remove Overlaps from WELFake

In [14]:
titles_to_remove = (
    exact_overlap_isot  | exact_overlap_eumix |
    fuzzy_overlap_isot  | fuzzy_overlap_eumix
)

n_before = len(welfake_clean)
welfake_clean = welfake_clean[
    ~welfake_clean["title_clean"].isin(titles_to_remove)
].reset_index(drop=True)
n_removed = n_before - len(welfake_clean)

print(f"Rows removed from WELFake: {n_removed:,}  ({100*n_removed/n_before:.3f}%)")
print(f"WELFake final row count:   {len(welfake_clean):,}")

Rows removed from WELFake: 38,737  (62.170%)
WELFake final row count:   23,571


## 4.4 Leakage Summary

In [15]:
def _decision(n):
    return "drop from WELFake" if n > 0 else "no action"

isot_total  = len(exact_overlap_isot  | fuzzy_overlap_isot)
eumix_total = len(exact_overlap_eumix | fuzzy_overlap_eumix)

summary = pd.DataFrame({
    "Pair":                  ["WELFake × ISOT", "WELFake × EU_mix"],
    "Exact matches":         [len(exact_overlap_isot),  len(exact_overlap_eumix)],
    "Fuzzy matches (J≥0.9)": [len(fuzzy_overlap_isot - exact_overlap_isot),
                               len(fuzzy_overlap_eumix - exact_overlap_eumix)],
    "Total removed":         [isot_total, eumix_total],
    "Decision":              [_decision(isot_total), _decision(eumix_total)],
})
print(summary.to_string(index=False))

welfake_clean.to_csv(OUTPUT_DIR / "welfake_clean.csv", index=False, encoding="utf-8")
print(f"\nwelfake_clean.csv saved: {len(welfake_clean):,} rows")

            Pair  Exact matches  Fuzzy matches (J≥0.9)  Total removed          Decision
  WELFake × ISOT          38729                      8          38737 drop from WELFake
WELFake × EU_mix             10                      1             11 drop from WELFake

welfake_clean.csv saved: 23,571 rows


## 4.5 ISOT × EU_mix Cross-Overlap

In [16]:
isot_ext_titles  = set(isot_clean["title_clean"].dropna())
eumix_ext_titles = set(eu_mix_clean["title_clean"].dropna())

exact_ext_overlap = isot_ext_titles & eumix_ext_titles
print(f"ISOT × EU_mix exact matches: {len(exact_ext_overlap):,}")

lsh_ext = MinHashLSH(threshold=0.9, num_perm=NUM_PERM)
for i, title in enumerate(isot_clean["title_clean"]):
    lsh_ext.insert(f"isot_{i}", make_minhash(title))

fuzzy_ext_overlap = set()
for title in eu_mix_clean["title_clean"]:
    if lsh_ext.query(make_minhash(title)):
        fuzzy_ext_overlap.add(title)

all_ext_overlap = exact_ext_overlap | fuzzy_ext_overlap
n_before_ext = len(eu_mix_clean)
eu_mix_clean = eu_mix_clean[
    ~eu_mix_clean["title_clean"].isin(all_ext_overlap)
].reset_index(drop=True)
n_removed_ext = n_before_ext - len(eu_mix_clean)

print(f"Rows removed from EU_mix: {n_removed_ext:,}")
print(f"EU_mix final row count:   {len(eu_mix_clean):,}")

eu_mix_clean.to_csv(OUTPUT_DIR / "eu_mix_clean.csv", index=False, encoding="utf-8")
print(f"eu_mix_clean.csv saved: {len(eu_mix_clean):,} rows")

ISOT × EU_mix exact matches: 10
Rows removed from EU_mix: 11
EU_mix final row count:   19,816
eu_mix_clean.csv saved: 19,816 rows


# 6. Train / Validation / Test Split

In [17]:
# 70/15/15 stratified split — same seeds as main notebook.
wf_train, wf_temp = train_test_split(
    welfake_clean, test_size=0.30, stratify=welfake_clean["label"], random_state=SEEDS[0]
)
wf_val, wf_test = train_test_split(
    wf_temp, test_size=0.50, stratify=wf_temp["label"], random_state=SEEDS[0]
)
wf_train = wf_train.reset_index(drop=True)
wf_val   = wf_val.reset_index(drop=True)
wf_test  = wf_test.reset_index(drop=True)

def _split_row(name, df):
    real = int((df["label"] == 0).sum())
    fake = int((df["label"] == 1).sum())
    return {"Split": name, "N": len(df), "Real (n)": real, "Fake (n)": fake,
            "Fake (%)": round(100 * fake / len(df), 2)}

summary = pd.DataFrame([
    _split_row("wf_train",     wf_train),
    _split_row("wf_val",       wf_val),
    _split_row("wf_test",      wf_test),
    _split_row("isot_clean",   isot_clean),
    _split_row("eu_mix_clean", eu_mix_clean),
])
print(summary.to_string(index=False))

assert len(wf_train) + len(wf_val) + len(wf_test) == len(welfake_clean)

wf_train.to_csv(OUTPUT_DIR / "wf_train.csv", index=False, encoding="utf-8")
wf_val.to_csv(OUTPUT_DIR   / "wf_val.csv",   index=False, encoding="utf-8")
wf_test.to_csv(OUTPUT_DIR  / "wf_test.csv",  index=False, encoding="utf-8")
print(f"\nSplit files saved to {OUTPUT_DIR.resolve()}")

       Split     N  Real (n)  Fake (n)  Fake (%)
    wf_train 16499      9507      6992     42.38
      wf_val  3536      2037      1499     42.39
     wf_test  3536      2038      1498     42.36
  isot_clean 38729     20826     17903     46.23
eu_mix_clean 19816      9963      9853     49.72

Split files saved to /Users/lukastefanovic/Desktop/Data Science II/outputs_no_norm


# 7. Feature Engineering

## 7.1 Lexical and Surface Features

In [18]:
# uppercase_ratio and titlecase_ratio use the original title — title_clean == title here,
# so the distinction is moot, but the code is kept identical to the main notebook.
def lexical_features(df: pd.DataFrame) -> pd.DataFrame:
    tc    = df["title_clean"].fillna("")
    title = df["title"].fillna("") if "title" in df.columns else tc

    tokens_clean = tc.str.split()
    tokens_orig  = title.str.split()

    word_count = tokens_clean.str.len().fillna(0)
    char_len   = tc.str.len()

    def _avg_word_len(tokens):
        return tokens.apply(lambda ws: sum(len(w) for w in ws) / len(ws) if ws else 0.0)
    def _type_token(tokens):
        return tokens.apply(lambda ws: len(set(ws)) / len(ws) if ws else 0.0)
    def _uppercase_ratio(tokens):
        return tokens.apply(
            lambda ws: sum(1 for w in ws if w.isupper() and len(w) >= 2) / len(ws) if ws else 0.0)
    def _titlecase_ratio(tokens):
        return tokens.apply(
            lambda ws: sum(1 for w in ws if w.istitle()) / len(ws) if ws else 0.0)

    has_number = tokens_clean.apply(
        lambda ws: int(any(any(c.isdigit() for c in w) for w in ws)) if ws else 0)

    return pd.DataFrame({
        "char_len":          char_len.values,
        "word_count":        word_count.values,
        "avg_word_len":      _avg_word_len(tokens_clean).values,
        "type_token_ratio":  _type_token(tokens_clean).values,
        "exclamation_count": tc.str.count(r"!").values,
        "question_count":    tc.str.count(r"\?").values,
        "ellipsis_count":    tc.str.count(r"\.\.\.").values,
        "has_number":        has_number.values,
        "uppercase_ratio":   _uppercase_ratio(tokens_orig).values,
        "titlecase_ratio":   _titlecase_ratio(tokens_orig).values,
    }, index=df.index)

lex_train  = lexical_features(wf_train)
lex_val    = lexical_features(wf_val)
lex_test   = lexical_features(wf_test)
lex_isot   = lexical_features(isot_clean)
lex_eumix  = lexical_features(eu_mix_clean)

print("lexical feature means (wf_train):")
print(lex_train.mean().round(4).to_string())

lexical feature means (wf_train):
char_len             72.5734
word_count           12.0660
avg_word_len          5.2196
type_token_ratio      0.9863
exclamation_count     0.0307
question_count        0.0644
ellipsis_count        0.0038
has_number            0.1466
uppercase_ratio       0.0350
titlecase_ratio       0.6730


## 7.2 Lexicon Overlap Features

In [19]:
_CLICKBAIT = frozenset([
    "you", "your", "this", "why", "what", "how", "top", "best", "worst",
    "secret", "shocking", "unbelievable", "amazing", "incredible", "breaking",
    "exclusive", "revealed", "truth", "real", "fake", "alert", "urgent",
    "must", "need", "watch", "see", "look", "just", "only", "never", "always",
    "everyone", "nobody", "everything", "nothing",
])
_HEDGING = frozenset([
    "may", "might", "could", "possibly", "perhaps", "apparently", "allegedly",
    "reportedly", "seems", "appears", "suggests", "claims", "according",
])
_MODAL = frozenset([
    "will", "would", "should", "shall", "must", "can", "could", "may", "might",
])

def lexicon_features(df: pd.DataFrame) -> pd.DataFrame:
    tc     = df["title_clean"].fillna("")
    tokens = tc.str.split()

    def _count(tok_series, word_set):
        return tok_series.apply(
            lambda ws: sum(1 for w in ws if w.lower() in word_set) if ws else 0)

    return pd.DataFrame({
        "clickbait_score": _count(tokens, _CLICKBAIT).values,
        "hedging_score":   _count(tokens, _HEDGING).values,
        "modal_count":     _count(tokens, _MODAL).values,
    }, index=df.index)

lex2_train  = lexicon_features(wf_train)
lex2_val    = lexicon_features(wf_val)
lex2_test   = lexicon_features(wf_test)
lex2_isot   = lexicon_features(isot_clean)
lex2_eumix  = lexicon_features(eu_mix_clean)

print("lexicon feature means (wf_train):")
print(lex2_train.mean().round(4).to_string())

lexicon feature means (wf_train):
clickbait_score    0.2250
hedging_score      0.0332
modal_count        0.0845


## 7.3 Semantic Features (VADER + Readability)

In [20]:
_vader = SentimentIntensityAnalyzer()

def semantic_features(df: pd.DataFrame) -> pd.DataFrame:
    tc = df["title_clean"].fillna("")

    vader_scores = df["title"].fillna("").apply(_vader.polarity_scores)
    compound = vader_scores.apply(lambda s: s["compound"])
    pos      = vader_scores.apply(lambda s: s["pos"])
    neg      = vader_scores.apply(lambda s: s["neg"])
    neu      = vader_scores.apply(lambda s: s["neu"])

    flesch = tc.apply(textstat.flesch_reading_ease)
    fog    = tc.apply(textstat.gunning_fog)

    return pd.DataFrame({
        "vader_compound":      compound.values,
        "vader_pos":           pos.values,
        "vader_neg":           neg.values,
        "vader_neu":           neu.values,
        "flesch_reading_ease": flesch.values,
        "gunning_fog":         fog.values,
    }, index=df.index)

sem_train  = semantic_features(wf_train)
sem_val    = semantic_features(wf_val)
sem_test   = semantic_features(wf_test)
sem_isot   = semantic_features(isot_clean)
sem_eumix  = semantic_features(eu_mix_clean)

print("mean vader_compound by label (wf_train):")
tmp = sem_train[["vader_compound"]].copy()
tmp["label"] = wf_train["label"].values
print(tmp.groupby("label")["vader_compound"].mean().rename({0: "real", 1: "fake"}).round(4).to_string())

mean vader_compound by label (wf_train):
label
real   -0.0807
fake   -0.0944


## 7.3.2 Named-Entity Features

In [21]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "senter"])

def ner_features(df):
    texts = df["title_clean"].fillna("").tolist()
    rows = []
    for doc in nlp.pipe(texts, batch_size=256):
        ents = [e.label_ for e in doc.ents]
        rows.append({
            "ner_person": ents.count("PERSON"),
            "ner_org":    ents.count("ORG"),
            "ner_gpe":    ents.count("GPE"),
            "ner_total":  len(ents),
        })
    return pd.DataFrame(rows, index=df.index)

ner_train = ner_features(wf_train)
ner_val   = ner_features(wf_val)
ner_test  = ner_features(wf_test)
ner_isot  = ner_features(isot_clean)
ner_eumix = ner_features(eu_mix_clean)

print("NER feature means (wf_train):")
print(ner_train.mean().round(4).to_string())

NER feature means (wf_train):
ner_person    0.5306
ner_org       0.5973
ner_gpe       0.2660
ner_total     1.8145


## 7.3.3 POS Distribution Features

In [22]:
def pos_features(df):
    texts = df["title_clean"].fillna("").tolist()
    rows = []
    for doc in nlp.pipe(texts, batch_size=256):
        total = max(len(doc), 1)
        tags  = [t.pos_ for t in doc]
        rows.append({
            "pos_noun_ratio":  tags.count("NOUN")  / total,
            "pos_verb_ratio":  tags.count("VERB")  / total,
            "pos_adj_ratio":   tags.count("ADJ")   / total,
            "pos_adv_ratio":   tags.count("ADV")   / total,
            "pos_propn_ratio": tags.count("PROPN") / total,
        })
    return pd.DataFrame(rows, index=df.index)

pos_train = pos_features(wf_train)
pos_val   = pos_features(wf_val)
pos_test  = pos_features(wf_test)
pos_isot  = pos_features(isot_clean)
pos_eumix = pos_features(eu_mix_clean)

print("POS feature means (wf_train):")
print(pos_train.mean().round(4).to_string())

POS feature means (wf_train):
pos_noun_ratio     0.0989
pos_verb_ratio     0.0788
pos_adj_ratio      0.0333
pos_adv_ratio      0.0144
pos_propn_ratio    0.4069


## 7.4 TF-IDF Vectorizers

In [23]:
# Fit on wf_train only; transform everything else with the fixed vocab.
tfidf_word = TfidfVectorizer(
    ngram_range=(1, 2), max_features=10_000, min_df=2, sublinear_tf=True
)
tfidf_char = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(3, 5), max_features=10_000, min_df=2, sublinear_tf=True
)

tfidf_word.fit(wf_train["title_clean"].fillna(""))
tfidf_char.fit(wf_train["title_clean"].fillna(""))

def _tf(vec, df):
    return vec.transform(df["title_clean"].fillna(""))

X_word_train = _tf(tfidf_word, wf_train)
X_word_val   = _tf(tfidf_word, wf_val)
X_word_test  = _tf(tfidf_word, wf_test)
X_word_isot  = _tf(tfidf_word, isot_clean)
X_word_eumix = _tf(tfidf_word, eu_mix_clean)

X_char_train = _tf(tfidf_char, wf_train)
X_char_val   = _tf(tfidf_char, wf_val)
X_char_test  = _tf(tfidf_char, wf_test)
X_char_isot  = _tf(tfidf_char, isot_clean)
X_char_eumix = _tf(tfidf_char, eu_mix_clean)

print(f"tfidf_word vocab: {len(tfidf_word.vocabulary_):,}")
print(f"tfidf_char vocab: {len(tfidf_char.vocabulary_):,}")
print(f"X_word_train shape: {X_word_train.shape}")

tfidf_word vocab: 10,000
tfidf_char vocab: 10,000
X_word_train shape: (16499, 10000)


## 7.5 Assembly of Feature Matrices

In [24]:
_splits_dense = [
    ("train",  wf_train,     lex_train,  lex2_train, sem_train, ner_train, pos_train),
    ("val",    wf_val,       lex_val,    lex2_val,   sem_val,   ner_val,   pos_val),
    ("test",   wf_test,      lex_test,   lex2_test,  sem_test,  ner_test,  pos_test),
    ("isot",   isot_clean,   lex_isot,   lex2_isot,  sem_isot,  ner_isot,  pos_isot),
    ("eumix",  eu_mix_clean, lex_eumix,  lex2_eumix, sem_eumix, ner_eumix, pos_eumix),
]

feat_frames = {}
for name, src_df, la, lb, lc, ld, le in _splits_dense:
    feat = pd.concat(
        [la.reset_index(drop=True), lb.reset_index(drop=True),
         lc.reset_index(drop=True), ld.reset_index(drop=True),
         le.reset_index(drop=True)],
        axis=1,
    )
    feat["label"] = src_df["label"].values
    feat_frames[name] = feat

feat_train = feat_frames["train"]
feat_val   = feat_frames["val"]
feat_test  = feat_frames["test"]
feat_isot  = feat_frames["isot"]
feat_eumix = feat_frames["eumix"]

for name, feat in feat_frames.items():
    feat.to_csv(OUTPUT_DIR / f"feat_{name}.csv", index=False)

with open(MODELS_DIR / "tfidf_word.pkl", "wb") as f:
    pickle.dump(tfidf_word, f)
with open(MODELS_DIR / "tfidf_char.pkl", "wb") as f:
    pickle.dump(tfidf_char, f)

rows_info = [{"Split": n, "Rows": len(f), "Features (dense)": f.shape[1] - 1}
             for n, f in feat_frames.items()]
print("Feature matrix provenance:")
print(pd.DataFrame(rows_info).to_string(index=False))

Feature matrix provenance:
Split  Rows  Features (dense)
train 16499                28
  val  3536                28
 test  3536                28
 isot 38729                28
eumix 19816                28


# 8. Modeling — Classical Models

In [25]:
# Accumulate per-model × dataset results here.
results = []

## 8.1 Logistic Regression (TF-IDF Word N-grams)

In [26]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

y_train = feat_train["label"].values

lr = LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs",
                        class_weight="balanced", random_state=42)
lr.fit(X_word_train, y_train)

for ds_name, X, y in [
    ("wf_val",   X_word_val,   feat_val["label"].values),
    ("wf_test",  X_word_test,  feat_test["label"].values),
    ("isot",     X_word_isot,  feat_isot["label"].values),
    ("eumix",    X_word_eumix, feat_eumix["label"].values),
]:
    yp   = lr.predict(X)
    prob = lr.predict_proba(X)[:, 1]
    results.append(dict(
        model="LogisticRegression", seed=42, dataset=ds_name,
        accuracy=accuracy_score(y, yp),
        macro_f1=f1_score(y, yp, average="macro"),
        roc_auc=roc_auc_score(y, prob),
    ))

with open(MODELS_DIR / "lr.pkl", "wb") as f:
    pickle.dump(lr, f)

val_f1 = next(r["macro_f1"] for r in results
              if r["model"] == "LogisticRegression" and r["dataset"] == "wf_val")
print(f"LR  val macro-F1: {val_f1:.4f}")

LR  val macro-F1: 0.8797


## 8.2 Multinomial Naive Bayes (TF-IDF Word N-grams)

In [27]:
y_train = feat_train["label"].values

nb_clf = MultinomialNB(alpha=0.1)
nb_clf.fit(X_word_train, y_train)

for ds_name, X, y in [
    ("wf_val",   X_word_val,   feat_val["label"].values),
    ("wf_test",  X_word_test,  feat_test["label"].values),
    ("isot",     X_word_isot,  feat_isot["label"].values),
    ("eumix",    X_word_eumix, feat_eumix["label"].values),
]:
    yp   = nb_clf.predict(X)
    prob = nb_clf.predict_proba(X)[:, 1]
    results.append(dict(
        model="NaiveBayes", seed=None, dataset=ds_name,
        accuracy=accuracy_score(y, yp),
        macro_f1=f1_score(y, yp, average="macro"),
        roc_auc=roc_auc_score(y, prob),
    ))

with open(MODELS_DIR / "nb.pkl", "wb") as f:
    pickle.dump(nb_clf, f)

val_f1 = next(r["macro_f1"] for r in results
              if r["model"] == "NaiveBayes" and r["dataset"] == "wf_val")
print(f"NB  val macro-F1: {val_f1:.4f}")

NB  val macro-F1: 0.8463


## 8.3 Linear SVM (TF-IDF Word + Character N-grams)

In [28]:
y_train = feat_train["label"].values

X_svm_train = sp.hstack([X_word_train, X_char_train])
X_svm_val   = sp.hstack([X_word_val,   X_char_val])
X_svm_test  = sp.hstack([X_word_test,  X_char_test])
X_svm_isot  = sp.hstack([X_word_isot,  X_char_isot])
X_svm_eumix = sp.hstack([X_word_eumix, X_char_eumix])

svc     = LinearSVC(C=1.0, class_weight="balanced", max_iter=2000)
svm_cal = CalibratedClassifierCV(svc, cv=3)
svm_cal.fit(X_svm_train, y_train)

for ds_name, X, y in [
    ("wf_val",   X_svm_val,   feat_val["label"].values),
    ("wf_test",  X_svm_test,  feat_test["label"].values),
    ("isot",     X_svm_isot,  feat_isot["label"].values),
    ("eumix",    X_svm_eumix, feat_eumix["label"].values),
]:
    yp   = svm_cal.predict(X)
    prob = svm_cal.predict_proba(X)[:, 1]
    results.append(dict(
        model="LinearSVM", seed=None, dataset=ds_name,
        accuracy=accuracy_score(y, yp),
        macro_f1=f1_score(y, yp, average="macro"),
        roc_auc=roc_auc_score(y, prob),
    ))

with open(MODELS_DIR / "svm.pkl", "wb") as f:
    pickle.dump(svm_cal, f)

val_f1 = next(r["macro_f1"] for r in results
              if r["model"] == "LinearSVM" and r["dataset"] == "wf_val")
print(f"SVM val macro-F1: {val_f1:.4f}")

SVM val macro-F1: 0.8930


## 8.4 Random Forest (Engineered Features)

In [29]:
X_rf_train = feat_train.drop("label", axis=1)
y_train    = feat_train["label"].values

rf = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                            random_state=42, n_jobs=-1)
rf.fit(X_rf_train, y_train)

for ds_name, df in [
    ("wf_val",   feat_val),
    ("wf_test",  feat_test),
    ("isot",     feat_isot),
    ("eumix",    feat_eumix),
]:
    y    = df["label"].values
    X    = df.drop("label", axis=1)
    yp   = rf.predict(X)
    prob = rf.predict_proba(X)[:, 1]
    results.append(dict(
        model="RandomForest", seed=42, dataset=ds_name,
        accuracy=accuracy_score(y, yp),
        macro_f1=f1_score(y, yp, average="macro"),
        roc_auc=roc_auc_score(y, prob),
    ))

with open(MODELS_DIR / "rf.pkl", "wb") as f:
    pickle.dump(rf, f)

val_f1 = next(r["macro_f1"] for r in results
              if r["model"] == "RandomForest" and r["dataset"] == "wf_val")
print(f"RF  val macro-F1: {val_f1:.4f}")

RF  val macro-F1: 0.7482


## 8.5 Threshold Tuning

In [30]:
import numpy as np

thresholds = np.arange(0.10, 0.91, 0.05)

def best_threshold(probs, labels):
    best_t, best_f1 = 0.5, -1
    for t in thresholds:
        preds = (probs >= t).astype(int)
        f1 = f1_score(labels, preds, average="macro")
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1

y_val  = feat_val["label"].values
y_test = feat_test["label"].values

prob_models = {
    "LogisticRegression": (
        lr.predict_proba(X_word_val)[:, 1],
        lr.predict_proba(X_word_test)[:, 1],
    ),
    "NaiveBayes": (
        nb_clf.predict_proba(X_word_val)[:, 1],
        nb_clf.predict_proba(X_word_test)[:, 1],
    ),
    "LinearSVM": (
        svm_cal.predict_proba(X_svm_val)[:, 1],
        svm_cal.predict_proba(X_svm_test)[:, 1],
    ),
    "RandomForest": (
        rf.predict_proba(feat_val.drop("label", axis=1))[:, 1],
        rf.predict_proba(feat_test.drop("label", axis=1))[:, 1],
    ),
}

opt_thresholds = {}
rows = []
for name, (p_val, p_test) in prob_models.items():
    t_opt, f1_val_opt = best_threshold(p_val, y_val)
    f1_val_def  = f1_score(y_val,  (p_val  >= 0.5).astype(int), average="macro")
    f1_test_opt = f1_score(y_test, (p_test >= t_opt).astype(int), average="macro")
    opt_thresholds[name] = t_opt
    rows.append({"model": name, "opt_threshold": round(t_opt, 2),
                 "val_f1@0.5": round(f1_val_def, 4),
                 "val_f1@opt": round(f1_val_opt, 4),
                 "test_f1@opt": round(f1_test_opt, 4)})

thresh_df = pd.DataFrame(rows)
print(thresh_df.to_string(index=False))
thresh_df.to_csv(OUTPUT_DIR / "threshold_results_no_norm.csv", index=False)
print(f"\nSaved to {OUTPUT_DIR / 'threshold_results_no_norm.csv'}")

             model  opt_threshold  val_f1@0.5  val_f1@opt  test_f1@opt
LogisticRegression           0.60      0.8797      0.8884       0.8733
        NaiveBayes           0.35      0.8463      0.8575       0.8488
         LinearSVM           0.45      0.8930      0.8956       0.8811
      RandomForest           0.45      0.7463      0.7490       0.7447

Saved to outputs_no_norm/threshold_results_no_norm.csv


## 8.6 Master Results Table

In [31]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_DIR / "results_master.csv", index=False)

pivot = (results_df
         .dropna(subset=["macro_f1"])
         .groupby(["model", "dataset"])["macro_f1"]
         .mean()
         .unstack("dataset")
         .round(3))

ds_order = [c for c in ["wf_val", "wf_test", "isot", "eumix"] if c in pivot.columns]
pivot = pivot[ds_order]
print("Macro-F1 by model and dataset:")
print(pivot.to_string())

print()
for ds in ["wf_test", "isot", "eumix"]:
    if ds in pivot.columns:
        best = pivot[ds].idxmax()
        print(f"Best on {ds}: {best}  (macro-F1={pivot.loc[best, ds]:.3f})")

Macro-F1 by model and dataset:
dataset             wf_val  wf_test   isot  eumix
model                                            
LinearSVM            0.893    0.882  0.653  0.327
LogisticRegression   0.880    0.866  0.572  0.325
NaiveBayes           0.846    0.824  0.649  0.283
RandomForest         0.748    0.750  0.737  0.490

Best on wf_test: LinearSVM  (macro-F1=0.882)
Best on isot: RandomForest  (macro-F1=0.737)
Best on eumix: RandomForest  (macro-F1=0.490)


<div style="background-color: #d4edda; border-left: 5px solid #28a745; padding: 12px; border-radius: 4px; color: black;">

**What is title normalisation, and why does it matter?**

Before feeding headlines into any model, Section 3.4 (from the main notebook) applied a cleaning pipeline to every title: all text was converted to lowercase, URLs and Twitter handles were stripped out, and for the ISOT dataset specifically, trailing source attributions such as "— Reuters" or "| BBC News" were removed. The cleaned version was stored as `title_clean`, which is the column all models were trained and evaluated on. The purpose of this step is to reduce noise — for example, ensuring that "TRUMP", "Trump", and "trump" are all treated as the same word by the model, rather than three separate tokens that happen to mean the same thing.

**What did we change in this experiment?**

In this notebook, we skipped that entire cleaning step. Instead, `title_clean` was set equal to the raw, unmodified title exactly as it appeared in the original dataset. This means models were trained and evaluated on headlines that may contain mixed capitalisation, source attributions, and any other raw formatting present in the data.

**What do the results show?**

Comparing the two runs, the differences in performance are small but informative:

- **TF-IDF models (Logistic Regression, Naive Bayes, LinearSVM):** These models showed almost no meaningful change when normalisation was removed. LinearSVM, for example, scored 0.890 on the validation set with normalisation and 0.893 without, which is a difference of under half a percentage point. This tells us that for models working with word frequencies, lowercasing and stripping URLs has very little practical effect, likely because genuine discriminative vocabulary (words that actually signal fake vs. real news) dominates the signal regardless of casing.

- **Random Forest:** This model was more sensitive to the change, dropping approximately 2.5 percentage points on the validation set (from 0.773 to 0.748) without normalisation. The Random Forest does not use word frequencies — it uses features that we crafted like word count, punctuation density, and named-entity counts. These features are computed from `title_clean`, so when that column contains raw, inconsistently cased text, the feature values become noisier and the model suffers slightly.

- **ISOT attribution stripping:** The most impactful part of the normalisation pipeline for ISOT was stripping the trailing source labels. Without this step, phrases like "— Reuters" appear as tokens in the ISOT titles, which could artificially inflate performance on that dataset by leaking publisher identity as a signal rather than genuine linguistic content.

**Overall conclusion**

Title normalisation is important for consistency and model fairness, but it is not the decisive factor in this pipeline's performance. The TF-IDF models are robust enough that removing it barely moves the needle. The Random Forest benefits slightly from it due to cleaner feature computation. The more important takeaway is that normalisation prevents the models from accidentally learning to recognise news sources rather than headline language, which is a subtle but meaningful form of data leakage that would overestimate real-world performance.

</div>